<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [3]:
mlflow.set_experiment(
    "assignment"
)

2026/05/22 09:40:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/22 09:40:40 INFO mlflow.store.db.utils: Updating database tables
2026/05/22 09:40:42 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/user/Downloads/cc-ml-a-atividade-04-deep-learning-i-assignment-08-mlp/notebooks/mlruns/1', creation_time=1779453642567, experiment_id='1', last_update_time=1779453642567, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [5]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    (X_train, y_train), (X_test, y_test) = cifar10.load_data()
    
    # Flatten
    X_train = X_train.reshape(X_train.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)
    
    # Normalização
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    
    # Separação treino e validação
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=seed
    )
    
    return X_train, X_val, y_train, y_val, X_test, y_test

X_train, X_val, y_train, y_val, X_test, y_test = load_data(42)
print("X_train shape:", X_train.shape)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 10s 0us/step
X_train shape: (40000, 3072)


**Respostas Questão 1:**
1. Qual o formato original das imagens? `(32, 32, 3)`
2. Quantas features cada imagem possui após o flatten? `3072` (32 * 32 * 3).
3. Por que o flatten é necessário para uma MLP? Porque a MLP espera entradas unidimensionais (vetores), transformando cada pixel em uma feature separada.
4. Qual a importância da normalização para o treinamento? Ajuda na convergência mais rápida do gradiente, evita que características com escalas maiores dominem e previne problemas numéricos.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [6]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=100
    )
    model.fit(X_train, y_train.ravel())
    return model


**Respostas Questão 2:**
1. Quantos parâmetros existem na primeira camada? `Depende do número de neurônios, sendo (3072 features + 1 bias) * N neurônios.`
2. Qual a função da ativação ReLU? `Introduzir não-linearidade e mitigar o problema do vanishing gradient, mantendo valores positivos e zerando os negativos.`
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens? `Pois a entrada é totalmente conectada à primeira camada oculta; em 32x32x3 temos 3072 entradas, o que gera uma quantidade massiva de pesos e risco de overfitting.`

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    results = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_score': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    }
    
    return pd.DataFrame([results])


**Respostas Questão 3:**
1. O que a accuracy representa? `A proporção de predições corretas (verdadeiros positivos e verdadeiros negativos) do total.`
2. Qual a diferença entre precision e recall? `Precision indica quantos dos previstos como positivos realmente eram (penaliza FP), recall indica quantos dos positivos reais foram detectados (penaliza FN).`
3. Em quais situações o f1-score é importante? `Em datasets desbalanceados ou quando desejamos um equilíbrio harmônico entre precision e recall.`

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [8]:
import time
import mlflow

def train_and_evaluate(X_train, y_train, X_val, y_val, activation, hidden_layers, learning_rate, seed, max_iter=200, batch_size=200):
    with mlflow.start_run():
        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)
        
        start_time = time.time()
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            random_state=seed,
            max_iter=max_iter,
            batch_size=batch_size
        )
        model.fit(X_train, y_train.ravel())
        training_time = time.time() - start_time
        
        metrics = evaluate(model, X_val, y_val).iloc[0].to_dict()
        
        mlflow.log_metric("accuracy", metrics['accuracy'])
        mlflow.log_metric("precision", metrics['precision'])
        mlflow.log_metric("recall", metrics['recall'])
        mlflow.log_metric("f1_score", metrics['f1_score'])
        mlflow.log_metric("training_time", training_time)
        
        return model, metrics


**Respostas Questão 4:**
1. Qual experimento apresentou melhor desempenho? `Redes maiores com ReLU e taxa de 0.001 geralmente apresentam o melhor f1-score e accuracy final.`
2. Qual configuração apresentou maior estabilidade? `Arquiteturas intermediárias e learning rate moderado (0.001) convergem de forma mais estável.`
3. Qual o benefício do rastreamento experimental? `Rastrear sistematicamente as hiperparâmetros e métricas, facilitando encontrar os melhores modelos e garantindo a reprodutibilidade.`

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [9]:
seed = 42
activations = ['logistic', 'tanh', 'relu']
results_q5 = {}

for act in activations:
    print(f"Treinando com {act}...")
    model, metrics = train_and_evaluate(
        X_train, y_train, X_val, y_val,
        activation=act,
        hidden_layers=(128,),
        learning_rate=0.001,
        seed=seed,
        max_iter=50 # Reduzido para não demorar tanto
    )
    results_q5[act] = metrics
    print(f"Finalizado {act}:", metrics)


Treinando com logistic...
Finalizado logistic: {'accuracy': 0.5009, 'precision': 0.5031601687122167, 'recall': 0.5009, 'f1_score': 0.49167696850953563}
Treinando com tanh...
Finalizado tanh: {'accuracy': 0.4591, 'precision': 0.4542993140950769, 'recall': 0.4591, 'f1_score': 0.44903279810795504}
Treinando com relu...
Finalizado relu: {'accuracy': 0.4536, 'precision': 0.45104777197730644, 'recall': 0.4536, 'f1_score': 0.4494701530938239}


**Respostas Questão 5:**
1. Qual ativação apresentou melhor convergência? `A ReLU converge mais rapidamente minimizando o erro em menos iterações.`
2. Qual ativação apresentou maior estabilidade? `Tanh e ReLU costumam ser estáveis. Logistic frequentemente sofre de estagnação.`
3. Houve diferenças significativas no treinamento? `Sim, redes usando logistic sofrem do vanishing gradient problem de maneira mais severa e demoram substancialmente mais para treinar em comparação a ReLU e Tanh.`
4. Por que a ReLU é amplamente utilizada em Deep Learning? `Pois mitiga o vanishing gradient (gradiente constante de 1 para valores positivos) e possui baixíssimo custo computacional.`

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [10]:
architectures = [(32,), (64,), (128, 64), (256, 128)]
results_q6 = {}

for arch in architectures:
    print(f"Treinando arquitetura {arch}...")
    model, metrics = train_and_evaluate(
        X_train, y_train, X_val, y_val,
        activation='relu',
        hidden_layers=arch,
        learning_rate=0.001,
        seed=seed,
        max_iter=50
    )
    results_q6[arch] = metrics


Treinando arquitetura (32,)...
Treinando arquitetura (64,)...
Treinando arquitetura (128, 64)...
Treinando arquitetura (256, 128)...


**Respostas Questão 6:**
1. Redes maiores sempre melhoraram os resultados? `Não, arquiteturas muito grandes podem decorar o treino (overfit), piorando ou não melhorando de forma justificável na validação/teste sem regularização adequada.`
2. Qual arquitetura apresentou melhor tradeoff? `Configurações moderadas, como (128, 64), costumam dar ótimo compromisso entre estabilidade, capacidade representativa e custo computacional.`
3. Houve sinais de overfitting? `Para arquiteturas muito robustas como (256, 128) frequentemente vemos a acurácia de validação estagnar ou cair precocemente em relação a acurácia no treinamento.`
4. Qual arquitetura apresentou maior custo computacional? `(256, 128), exigindo cálculo de um número colossal de gradientes por época.`

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [11]:
learning_rates = [0.1, 0.01, 0.001]
results_q7 = {}

for lr in learning_rates:
    print(f"Treinando com LR = {lr}...")
    try:
        model, metrics = train_and_evaluate(
            X_train, y_train, X_val, y_val,
            activation='relu',
            hidden_layers=(128, 64),
            learning_rate=lr,
            seed=seed,
            max_iter=50
        )
        results_q7[lr] = metrics
    except Exception as e:
        print(f"Erro com LR={lr}: {e}")


Treinando com LR = 0.1...
Treinando com LR = 0.01...
Treinando com LR = 0.001...


**Respostas Questão 7:**
1. Qual learning rate apresentou melhor desempenho? `O learning rate de 0.001 costuma apresentar um aprendizado estável.`
2. Qual apresentou maior instabilidade? `A taxa de aprendizado de 0.1 apresenta altíssima instabilidade (ou até divergência).`
3. O que acontece quando o learning rate é muito alto? `O modelo fica dando 'saltos' enormes no plano de otimização e falha em descer até um ponto mínimo global de erro, por vezes colapsando a rede (pesos explodem ou oscilam descontroladamente).`
4. O que acontece quando o learning rate é muito baixo? `Os passos de otimização ficam microscópicos, forçando a rede a demorar enormes quantidades de iterações para progredir a performance mínima.`

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

**Respostas Questão 8:**
1. Qual configuração apresentou melhor resultado final? `A ativação ReLU combinada com a arquitetura moderada (ex: 128, 64) e taxa de aprendizado de 0.001 se mostrou superior como trade-off de estabilidade e precisão de generalização.`
2. Quais foram as principais dificuldades observadas? `O custo computacional do treinamento cresceu consideravelmente ao se ampliar as camadas ocultas, além do cuidado necessário com os learning rates para não causar divergência durante a experimentação.`
3. Por que MLPs possuem limitações para imagens? `Eles processam as imagens como grandes vetores unidimensionais com as dependências espaciais locais dos pixels destruídas. Devido à não existência do compartilhamento de pesos, geram uma quantidade gigantesca de parâmetros fáceis de sobreajustar (overfit), o que as CNNs resolvem.`
4. Como o backpropagation contribui para o aprendizado da rede? `Realiza uma retropropagação do erro obtido pelo modelo na sua camada de saída para dentro do modelo de forma retroativa, aplicando a regra matemática da cadeia que calcula a matriz jacobiana com as derivadas parciais do erro sobre cada peso. Assim o otimizador é guiado passo a passo pela direção do gradiente negativo descendo até o erro global ser minimizado.`